# RL

# Setup

In [ ]:
from thesis_project import (
    config,
)

config.default()

TICKER: config.FutTicker = "fbtp"

# Lobs directories
LOB_DIR = config.DATA_RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "fut"
LOB_CTD_DIR = LOB_DIR / "ctd"

# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.RES_FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.RES_TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_EXP_DIR = config.RES_EXP_DIR / TICKER / NOTEBOOK
    OUT_EXP_DIR.mkdir(parents=True, exist_ok=True)

# Common

In [ ]:
import json

from thesis_project.rl import report

# Main

## Baselines

In [ ]:
from thesis_project import rl
from thesis_project.dataset.features_market import FeatureSpec
from thesis_project.dataset.features_transforms import Identity
from thesis_project.rl.env_factory import make_gym_env

market: list[FeatureSpec] = [
    FeatureSpec(base_id="basis", transforms=[Identity()]),
]

dataset_config = rl.dataset.DatasetConfig(
    ticker="fbtp",
    n_jobs_market=1,
    contract_mode="round",
    market_set="dummy",
    calendar_set="dummy",
    calendar_enc_set="dummy",
)

dummy_dataset = rl.dataset.build_rl_dataset(dataset_config)

In [ ]:
env_config_serial = rl.env.EnvConfig(
    reset_mode="serial",
    price_mode="quoted",
    persistence_min=10,
    trajectory_min=0,
)

### Long Only

In [ ]:
env = make_gym_env(
    dummy_dataset,
    env_config_serial,
)

fixed_long_records = rl.evaluation.evaluate_model_gym(
    predictor=rl.policies.FixedLongPolicy(),
    env=env,
)

In [ ]:
metadata = {"which": "both"}
report.report({0: fixed_long_records}, metadata=metadata)

### Short Only

In [ ]:
env = make_gym_env(dummy_dataset, env_config_serial)

fixed_short_records = rl.evaluation.evaluate_model_gym(
    predictor=rl.policies.FixedShortPolicy(),
    env=env,
)

In [ ]:
metadata = {"which": "both"}
report.report({0: fixed_short_records}, metadata=metadata)

### Random

In [ ]:
import pandas as pd

from thesis_project.utils.misc import generate_seeds

seed = 42
seeds = generate_seeds(n=5, seed=seed)

random_records: dict[int, pd.DataFrame] = {}
for seed in seeds:
    env = make_gym_env(dataset=dummy_dataset, env_config=env_config_serial)
    policy = rl.policies.RandomPolicy(seed=seed)
    random_records[seed] = rl.evaluation.evaluate_model_gym(predictor=policy, env=env)

In [ ]:
metadata = {"which": "both"}
report.report(random_records, metadata=metadata)

## PPO

In [ ]:
# TODO:
# - Try to normalize data z-score
# - Try 2x, 5x, 10x CTD
# - Try different persistence: 1min, 30min, 60min

# NOTE:
# - Run 3 -> Mid Prices
# - Run 4 -> Quoted Prices

root = config.RES_EXP_DIR / TICKER / "rl"
run = 5
folder = f"run-{run:03d}"
path = root / folder

with open(path / "config.json", "r") as f:
    settings = json.load(f)

seeds = settings["seeds"]

### Training set

In [ ]:
records = report.load_records(
    path,
    batch_size=64,
    clip_range=0.2,
    seeds=seeds,
    split="train",
)

In [ ]:
report.report(records)

### Test Set

In [ ]:
records = report.load_records(
    path,
    batch_size=64,
    clip_range=0.2,
    seeds=seeds,
    split="test",
)

In [ ]:
report.report(records)

### Spread considerations

In [ ]:
from thesis_project.dataset.data import BASE_FEATURES
from thesis_project.utils.io import load_filtered_parquet

pnl_mean = 76_439.38
pnl_std = 17_757.55
n_trades_mean = 786.20
n_trades_std = 366.64

max_spread = pnl_mean / n_trades_mean

spread_basis = records[seeds[0]].set_index("timestamp")["basis_spread"]

spread_ctd = load_filtered_parquet(BASE_FEATURES["ctd_spread"].path).loc[spread_basis.index]
spread_ctd *= 2_000_000 / 100
spread_fut = load_filtered_parquet(BASE_FEATURES["fut_spread"].path).loc[spread_basis.index]
spread_fut *= 100_000 / 100

In [ ]:
n = 22
print(f"{'Max spread per trade:':<{n}} {max_spread:,.2f} €")
print(f"{'Basis spread:':<{n}} {spread_basis.mean() / 2:,.2f} ± {spread_basis.std() / 2:,.2f} €")
print(f"{'FBTP CTD spread:':<{n}} {spread_ctd.mean() / 2:,.2f} ± {spread_ctd.std() / 2:,.2f} €")
print(
    f"{'FBTP FUT spread:':<{n}} {15.5 * spread_fut.mean() / 2:,.2f} ± {18 * spread_fut.std() / 2:,.2f} €"
)

### Other

In [ ]:
import pandas as pd


def check_evaluation_records(
    records: pd.DataFrame,
    *,
    expected_episodes: int | None = None,
    expected_terminal_time: str = "17:00:00",
) -> None:
    if records.empty:
        raise AssertionError("Evaluation records are empty")

    display(records.head(5))

    required_columns = {
        "episode",
        "timestamp",
        "step",
        "position",
        "reward",
        "terminal",
    }
    missing_columns = required_columns - set(records.columns)

    if missing_columns:
        raise AssertionError(f"Missing required columns: {sorted(missing_columns)}")

    episode_dates = records.groupby("episode")["timestamp"].first().dt.normalize()

    episode_lengths = records.groupby("episode").size()

    terminal_counts = records.groupby("episode")["terminal"].sum()

    terminal_rows = records[records["terminal"]]

    step_monotonic = (
        records.groupby("episode")["step"].apply(lambda s: s.is_monotonic_increasing).all()
    )

    dates_per_episode = records.groupby("episode")["timestamp"].apply(
        lambda s: s.dt.normalize().nunique()
    )

    duplicate_timestamps = records["timestamp"].duplicated().sum()

    terminal_time = pd.Timestamp(expected_terminal_time).time()

    checks = {
        "non-empty records": not records.empty,
        "no duplicate timestamps": duplicate_timestamps == 0,
        "terminal dtype is bool": records["terminal"].dtype == bool,
        "one terminal per episode": terminal_counts.eq(1).all(),
        "terminal count matches episodes": (
            records["terminal"].sum() == records["episode"].nunique()
        ),
        "one date per episode": dates_per_episode.eq(1).all(),
        "episode dates are unique": (episode_dates.nunique() == records["episode"].nunique()),
        "episode dates are chronological": episode_dates.is_monotonic_increasing,
        "steps are monotonic": step_monotonic,
        "terminal time is correct": (terminal_rows["timestamp"].dt.time.eq(terminal_time).all()),
    }

    if expected_episodes is not None:
        checks["expected episode count"] = records["episode"].nunique() == expected_episodes

    failed = [name for name, passed in checks.items() if not passed]

    print("Evaluation records validation")
    print("-" * 32)

    for name, passed in checks.items():
        status = "OK" if passed else "FAIL"
        print(f"{status:<5} {name}")

    print()
    print("Summary")
    print("-" * 32)
    print(f"Rows:                  {len(records):,}")
    print(f"Unique timestamps:     {records['timestamp'].nunique():,}")
    print(f"Duplicate timestamps:  {duplicate_timestamps:,}")
    print(f"Episodes:              {records['episode'].nunique():,}")
    print(f"Terminal observations: {records['terminal'].sum():,}")

    print()
    print("Episode lengths")
    print("-" * 32)
    print(f"Min:                   {episode_lengths.min():,}")
    print(f"Max:                   {episode_lengths.max():,}")
    print(f"Mean:                  {episode_lengths.mean():.1f}")

    if failed:
        raise AssertionError("Evaluation validation failed: " + ", ".join(failed))

    print()
    print("All checks passed.")